In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

In [2]:
df=pd.read_csv(r'C:\Users\shoba\stock_analysis\data\processed\cleaned_data.csv')
df['Date']=pd.to_datetime(df['Date'])
df=df.reset_index()
df=df.sort_values(by=['Date','Ticker'])

tickers=df['Ticker'].unique()

In [3]:
## RSI calculation
def calculate_rsi(data):
    delta=data['Close'].diff()
    delta=delta.reset_index(drop=True)

    gain = pd.Series(np.where(delta>0,delta,0))
    loss = pd.Series(np.where(delta<0,abs(delta),0))

    avg_gain = gain.ewm(span=14, min_periods=14).mean()
    avg_loss = loss.ewm(span=14, min_periods=14).mean()

    Rs = avg_gain/(avg_loss + 1e-10)

    RSI= round(100-(100/(1+Rs)),2)

    data['RSI'] = RSI

    return data['RSI']

In [4]:
rsi_data=[]

for ticker in tickers:
    stock_data = df[df['Ticker']==ticker].copy()
    stock_data=stock_data.reset_index(drop=True)
    stock_rsi=calculate_rsi(stock_data)
    stock_data['RSI'] = stock_rsi
    rsi_data.append(stock_data)

    stock_df=pd.concat(rsi_data)

stock_df.head(20)
stock_df.shape
stock_df.isnull().sum()
stock_df['RSI'].unique()

array([  nan, 66.95, 68.77, ..., 87.56, 14.5 , 11.9 ])

In [5]:
##RSI

rsi_signal=stock_df.groupby('Ticker').last().reset_index()

rsi_signal['signal'] = np.where (rsi_signal['RSI']<30,"🟢 BUY",np.where (rsi_signal['RSI']>70,"🔴 SELL","⚪HOLD"))
rsi_signal=rsi_signal.reset_index()
rsi_signal=rsi_signal.drop(columns=['level_0','index'])
rsi_signal.head(20)

,Ticker,Date,Company,Sector,Open,High,Low,Close,Volume,daily_return,year,month,day_of_week,RSI,signal
0,ADANIENT.NS,2026-06-12,Adani Enterprises,Conglomerate,2958.800049,2971.899902,2886.199951,2921.600098,2321009,0.004850,2026,6,4,52.38,⚪HOLD
1,ADANIPORTS.NS,2026-06-12,Adani Ports,Infrastructure,1804.000000,1818.000000,1777.900024,1812.900024,2325897,0.018712,2026,6,4,55.01,⚪HOLD
2,APOLLOHOSP.NS,2026-06-12,Apollo Hospitals,Healthcare,8535.000000,8558.000000,8440.000000,8498.000000,276444,0.000589,2026,6,4,70.57,🔴 SELL
3,ASIANPAINT.NS,2026-06-12,Asian Paints,Consumer,2725.699951,2753.000000,2709.399902,2747.399902,1078025,0.020997,2026,6,4,71.58,🔴 SELL
4,AXISBANK.NS,2026-06-12,Axis Bank,Banking,1332.000000,1358.500000,1318.500000,1356.300049,8535670,0.029606,2026,6,4,79.80,🔴 SELL
5,BAJAJ-AUTO.NS,2026-06-12,Bajaj Auto,Auto,10226.000000,10250.000000,9922.000000,10063.000000,386190,-0.005043,2026,6,4,30.43,⚪HOLD
6,BAJAJFINSV.NS,2026-06-12,Bajaj Finserv,Finance,1669.000000,1692.000000,1649.199951,1689.099976,1486234,0.026746,2026,6,4,41.58,⚪HOLD
7,BAJFINANCE.NS,2026-06-12,Bajaj Finance,Finance,881.200012,921.000000,881.099976,918.299988,11729668,0.054850,2026,6,4,61.01,⚪HOLD
8,BEL.NS,2026-06-12,BEL,Defence,407.350006,409.000000,403.299988,406.500000,8086130,0.010440,2026,6,4,39.39,⚪HOLD
9,BHARTIARTL.NS,2026-06-12,Bharti Airtel,Telecom,1784.000000,1825.000000,1784.000000,1822.500000,5910557,0.022383,2026,6,4,52.63,⚪HOLD


In [6]:
dropdown=widgets.Dropdown(
    options=sorted(df['Ticker'].unique()),
    description='Select Stock'
)

def plot_rsi(ticker):
    rsi_plot=stock_df[stock_df['Ticker']== ticker].copy()
    rsi_plot['Date'] = pd.to_datetime(rsi_plot['Date'])
    rsi_plot= rsi_plot.sort_values(by='Date')

    fig, (ax1, ax2) = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(14,8),
    sharex=True,
    height_ratios=[2,1]
    )

    rsi_plot.plot('Date', 'Close', color='blue',ax=ax1)
    rsi_plot.plot('Date', 'RSI', color='purple',ax=ax2)

    ax2.axhline(70, color='red', linestyle='--')
    ax2.axhline(50, color='gray', linestyle='--')
    ax2.axhline(30, color='green', linestyle='--')
    
    ax2.fill_between(
    rsi_plot['Date'],
    rsi_plot['RSI'],
    70,
    where=rsi_plot['RSI'] >= 70,
    color='red',
    alpha=0.3
    )
    ax2.fill_between(
    rsi_plot['Date'],
    rsi_plot['RSI'],
    30,
    where=rsi_plot['RSI'] <= 30,
    color='green',
    alpha=0.3
    )

    ax2.set_ylim(0,100)
    plt.tight_layout()
    plt.show()

widgets.interact(plot_rsi, ticker=dropdown)

interactive(children=(Dropdown(description='Select Stock', options=('ADANIENT.NS', 'ADANIPORTS.NS', 'APOLLOHOS…

<function __main__.plot_rsi(ticker)>

In [7]:
##MACD

def calculate_macd(data):
    ema12=data['Close'].ewm(span=12, min_periods=12).mean()
    ema26=data['Close'].ewm(span=26, min_periods=26).mean()

    macd_line=ema12-ema26
    signal_line=macd_line.ewm(span=9,min_periods=9).mean()

    histogram=macd_line-signal_line

    data['macd_line']=macd_line
    data['signal_line']=signal_line
    data['histogram']=histogram
    return data

macd_data=[]

for ticker in tickers:
    stock_data=df[df['Ticker']==ticker].copy()
    stock_data=stock_data.reset_index(drop=True)
    stock_macd=calculate_macd(stock_data)
    macd_data.append(stock_macd)

macd_df=pd.concat(macd_data)
macd_df=macd_df.reset_index(drop=True)

macd_df.shape

(60414, 17)

In [8]:
dropdown=widgets.Dropdown(
    options=sorted(df['Ticker'].unique()),
    description='Select Stock'
)


def plot_macd(ticker):
    macd_plot=macd_df[macd_df['Ticker']== ticker].copy()

    fig, (ax1, ax2, ax3) = plt.subplots(
    nrows=3, ncols=1,
    figsize=(14, 10),
    sharex=True,
    height_ratios=[3, 2, 1]
    )

    macd_plot.plot('Date','Close',color='blue',title=ticker,ax=ax1)
    macd_plot.plot('Date','macd_line',color='blue',ax=ax2)
    macd_plot.plot('Date','signal_line',color='orange',ax=ax2)
    ax3.bar(macd_df['Date'],macd_df['histogram'],color=np.where(macd_df['histogram'] >= 0, 'green', 'red'))

    ax2.axhline(0, color='gray', linestyle='--')
    ax2.legend(['MACD Line', 'Signal Line'])

    plt.tight_layout()
    plt.show()

widgets.interact(plot_macd, ticker=dropdown)

interactive(children=(Dropdown(description='Select Stock', options=('ADANIENT.NS', 'ADANIPORTS.NS', 'APOLLOHOS…

<function __main__.plot_macd(ticker)>

In [9]:
##Bollinger Bands

def calculate_bb(data):
    middle=data['Close'].rolling(window=20).mean()
    std=data['Close'].rolling(window=20).std()
    upper= middle + (2*std)
    lower= middle - (2*std)
    bandwidth = (upper - lower) / middle*100
    data['bb_middle']=middle
    data['bb_upper']=upper
    data['bb_lower']=lower
    data['bb_bandwidth']=bandwidth
    return data

bb_lst=[]

for ticker in tickers:
    bb_data=macd_df[macd_df['Ticker']==ticker].copy()
    bb_data=bb_data.reset_index(drop=True)
    bb_data=calculate_bb(bb_data)
    bb_lst.append(bb_data)

bb_df=pd.concat(bb_lst)
bb_df=bb_df.reset_index(drop=True)

bb_df.columns

    

Index(['index', 'Date', 'Ticker', 'Company', 'Sector', 'Open', 'High', 'Low',
       'Close', 'Volume', 'daily_return', 'year', 'month', 'day_of_week',
       'macd_line', 'signal_line', 'histogram', 'bb_middle', 'bb_upper',
       'bb_lower', 'bb_bandwidth'],
      dtype='object')

In [10]:
dropdown=widgets.Dropdown(
    options=sorted(df['Ticker'].unique()),
    description='Select Stock'
)

def plot_bb(ticker):
    bb_plot=bb_df[bb_df['Ticker']== ticker].copy()

    fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1,figsize=(14, 10),sharex=True)
   
    bb_plot.plot('Date','Close',color='blue',title=ticker,ax=ax1)
    bb_plot.plot('Date','bb_middle',color='orange',linestyle='--',ax=ax1)
    bb_plot.plot('Date','bb_upper',color='Red',ax=ax1)
    bb_plot.plot('Date','bb_lower',color='Green',ax=ax1)
   
    ax1.fill_between(bb_plot['Date'],bb_plot['bb_upper'],bb_plot['bb_lower'],color='lightblue',alpha=0.3)

    low_band=sorted(bb_plot['bb_bandwidth'])
    bb_plot.plot('Date','bb_bandwidth',color='purple',ax=ax2)
    ax2.axhline(low_band[0],color='gray',linestyle='--')
  
    plt.tight_layout()
    plt.show()

widgets.interact(plot_bb, ticker=dropdown)

interactive(children=(Dropdown(description='Select Stock', options=('ADANIENT.NS', 'ADANIPORTS.NS', 'APOLLOHOS…

<function __main__.plot_bb(ticker)>

In [11]:
##signal system

#merging RSI with macd and bb in final_df
final_df=bb_df.copy()
final_df = final_df.merge(stock_df[['Ticker', 'Date', 'RSI']],on=['Ticker', 'Date'],how='left')
final_df=final_df.reset_index(drop=True)
final_df['Date']=pd.to_datetime(final_df['Date'])
final_df=final_df.reset_index(drop=True)
final_df=final_df.drop(columns=['index'])
final_df.columns

#rsi score
rsi_conditions = [
    final_df['RSI'] < 30,
    final_df['RSI'] > 70
]
rsi_choices = [1, -1]

final_df['rsi_score'] = np.select(rsi_conditions, rsi_choices, default=0)

#MACD score
macd_conditions = [
    final_df['histogram'] > 0,
    final_df['histogram'] < 0
]
macd_choices = [1, -1]

final_df['macd_score'] = np.select(macd_conditions, macd_choices, default=0)

#BB score
bb_conditions = [
    final_df['Close'] < final_df['bb_lower'],
    final_df['Close'] > final_df['bb_upper']
]
bb_choices = [1, -1]

final_df['bb_score'] = np.select(bb_conditions, bb_choices, default=0)

#total score
final_df['total_score'] = (
    final_df['rsi_score'] +
    final_df['macd_score'] +
    final_df['bb_score']
)

#Final signal
signal_conditions = [
    final_df['total_score'] >= 2,
    final_df['total_score'] == 1,
    final_df['total_score'] == 0,
    final_df['total_score'] == -1,
    final_df['total_score'] <= -2
]
signal_choices = [
    'STRONG BUY 🟢',
    'WEAK BUY 🔵',
    'NEUTRAL ⚪',
    'WEAK SELL 🟡',
    'STRONG SELL 🔴'
]

final_df['final_signal'] = np.select(
    signal_conditions,
    signal_choices,
    default='NEUTRAL ⚪'
)
    
#current signal for each stock
current_signal = final_df.groupby('Ticker').last().reset_index()
current_signal = current_signal[[
    'Ticker','RSI','rsi_score',
    'macd_score','bb_score',
    'total_score','final_signal'
]]
current_signal = current_signal.sort_values('total_score', ascending=False)
current_signal = current_signal.reset_index(drop=True)
current_signal

#csv file
final_df.to_csv(
    r'C:\\Users\\shoba\\stock_analysis\\data\\processed\\final_df.csv',
    index=False
)
current_signal.to_csv(
    r'C:\\Users\\shoba\\stock_analysis\\data\\processed\\current_signal.csv',
    index=False
)